# NeuralMath — तंत्रिका नेटवर्क की गणितीय संरचना

हम एक सरल वर्गीकरण समस्या के माध्यम से कृत्रिम तंत्रिका नेटवर्क का गणितीय परिचय प्रस्तुत करते हैं। एक रैखिक वर्गीकारक स्वाभाविक रूप से कृत्रिम न्यूरॉन की अवधारणा तक ले जाता है; इसके बाद हम भार, बायस, सक्रियण फलनों और अनेक न्यूरॉनों को एक छोटे नेटवर्क में संयोजित करने की प्रक्रिया पर चर्चा करते हैं। प्रशिक्षण को हानि फलनों, अवकलजों, ग्रेडिएंट अवरोह और प्रतिप्रसारण के माध्यम से प्रस्तुत किया जाता है। अंत में मॉडल को सीधे Python में लागू किया जाता है, जिससे समीकरणों और एल्गोरिथ्म के बीच का संबंध स्पष्ट हो जाता है। उद्देश्य यह दिखाना है कि विश्लेषणात्मक ज्यामिति, रैखिक बीजगणित और कलन की परिचित अवधारणाएँ तंत्रिका नेटवर्क की मूल कार्यप्रणाली को समझने के लिए एक सुलभ आधार प्रदान करती हैं।

**NeuralMath.org** · *A anatomia matemática de uma rede neural*

चर और फ़ंक्शन के नाम अंग्रेज़ी में रखे गए हैं; व्याख्यात्मक पाठ इस पृष्ठ की भाषा में है।


## 1. एक छोटा तंत्रिका नेटवर्क बनाना

चर और फ़ंक्शन के नाम अंग्रेज़ी में रखे गए हैं; व्याख्यात्मक पाठ इस पृष्ठ की भाषा में है।

$$\mathbf{z}^{(1)}=W^{(1)}\mathbf{x}+\mathbf{b}^{(1)},\qquad \mathbf{h}=\operatorname{ReLU}(\mathbf{z}^{(1)}),$$

$$z^{(2)}=W^{(2)}\mathbf{h}+b^{(2)},\qquad \widehat y=\sigma(z^{(2)}).$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def relu(s):
    return np.maximum(0, s)


def relu_derivative(s):
    return (s > 0).astype(float)


def sigmoid(s):
    return 1 / (1 + np.exp(-s))


def base_boundary(m):
    return (
        6.68
        + 0.74 * np.exp(-((m - 155) / 20) ** 2)
        + 1.50 / (1 + np.exp(-(m - 202) / 5))
    )


## 2. समस्या का गणितीय निरूपण

तंत्रिका नेटवर्क कैसे काम करता है, यह समझने के लिए हम जानबूझकर एक सरल समस्या से शुरुआत करते हैं: सेब और संतरे में अंतर करना। किसी फल में रंग, द्रव्यमान, व्यास, बनावट और सुगंध जैसी अनेक विशेषताएँ हो सकती हैं, जिनका उपयोग इस कार्य में किया जा सकता है। किंतु एक प्रारंभिक गणितीय मॉडल को यह सारी जटिलता शामिल करने की आवश्यकता नहीं है। हम केवल दो ऐसी राशियाँ चुनेंगे जिन्हें आसानी से मापा जा सकता है: ग्राम में द्रव्यमान और सेंटीमीटर में व्यास।

$$x_j^{\ast}=\frac{x_j-\mu_j}{\sigma_j}.$$


In [ ]:
def build_dataset():
    masses_1 = np.linspace(118, 212, 16)
    masses_2 = masses_1 + np.array([
        1.2, -1.1, 0.8, -1.3, 1.0, -0.8, 1.1, -1.0,
        1.3, -0.9, 0.9, -1.2, 1.0, -1.0, 0.8, -0.7,
    ])
    k = np.arange(16)

    apple_1 = np.column_stack([
        masses_1,
        base_boundary(masses_1) + 0.22 + 0.05 * np.sin(0.8 * k),
    ])
    apple_2 = np.column_stack([
        masses_2,
        base_boundary(masses_2) + 0.58 + 0.06 * np.cos(0.45 * k),
    ])
    orange_1 = np.column_stack([
        masses_1,
        base_boundary(masses_1) - (0.22 + 0.04 * np.cos(0.7 * k)),
    ])
    orange_2 = np.column_stack([
        masses_2,
        base_boundary(masses_2) - (0.55 + 0.05 * np.sin(0.5 * k)),
    ])

    X_train = np.vstack([apple_1, apple_2, orange_1, orange_2])
    y_train = np.array([1.0] * 32 + [0.0] * 32)

    X_test = np.array([
        [121.0, 7.20], [135.0, 7.45], [152.0, 7.75], [171.0, 7.32],
        [121.0, 6.28], [152.0, 6.86], [189.0, 6.48], [208.0, 7.55],
    ])
    y_test = np.array([1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0])
    return X_train, y_train, X_test, y_test


X_train, y_train, X_test, y_test = build_dataset()
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
Xn = (X_train - mean) / std

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('mean:', mean)
print('std:', std)


## 3. तंत्रिका नेटवर्क कैसे सीखता है?

भार और बायस पहले से ज्ञात नहीं होते। उन्हें ऐसे उदाहरणों से समायोजित करना पड़ता है जिनके सही उत्तर उपलब्ध हों। इस प्रक्रिया को प्रशिक्षण कहते हैं और इसे इस चक्र से संक्षेपित किया जा सकता है \[\boxed{\text{पूर्वानुमान}}
\longrightarrow
\boxed{\text{त्रुटि मापें}}
\longrightarrow
\boxed{\text{पैरामीटर समायोजित करें}}
\longrightarrow
\boxed{\text{दोहराएँ}}.\]


In [ ]:
rng = np.random.default_rng(42)
W1 = 0.1 * rng.standard_normal((12, 2))
b1 = np.zeros(12)
W2 = 0.1 * rng.standard_normal(12)
b2 = 0.0

learning_rate = 0.05
n_epochs = 6000
history = []

for epoch in range(n_epochs):
    for i in rng.permutation(len(Xn)):
        x = Xn[i]
        target = y_train[i]

        z1 = W1 @ x + b1
        h = relu(z1)
        z2 = W2 @ h + b2
        y_hat = sigmoid(z2)

        delta2 = (y_hat - target) * y_hat * (1 - y_hat)
        dW2 = delta2 * h
        db2 = delta2

        delta1 = (delta2 * W2) * relu_derivative(z1)
        dW1 = np.outer(delta1, x)
        db1 = delta1

        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2

    errors = []
    for x, target in zip(Xn, y_train):
        h = relu(W1 @ x + b1)
        y_hat = sigmoid(W2 @ h + b2)
        errors.append(0.5 * (target - y_hat) ** 2)
    history.append(float(np.mean(errors)))

print(f"E_final: {history[-1]:.10f}")


## 4. गणित से Python तक

अब हम समीकरणों से एक चलने योग्य प्रोग्राम की ओर बढ़ते हैं। उद्देश्य किसी विशेष तंत्रिका-नेटवर्क लाइब्रेरी का उपयोग करना नहीं, बल्कि NumPy के साथ लेख में विकसित गणितीय क्रियाओं को सीधे लागू करना है । इससे पंक्ति-दर-पंक्ति देखा जा सकता है कि रैखिक संयोजन, सक्रियण फलन, हानि की गणना, प्रतिप्रसारण और ग्रेडिएंट अवरोह कोड में कैसे दिखाई देते हैं।


In [ ]:
def predict(X):
    X_scaled = (X - mean) / std
    outputs = []
    for x in X_scaled:
        h = relu(W1 @ x + b1)
        outputs.append(sigmoid(W2 @ h + b2))
    return np.array(outputs)


pred_train = predict(X_train)
pred_test = predict(X_test)

print(f"acc_train: {np.mean((pred_train >= 0.5) == y_train):.4f}")
print(f"acc_test: {np.mean((pred_test >= 0.5) == y_test):.4f}")
print('y_hat_test:', np.round(pred_test, 6))

masses = np.linspace(116, 214, 500)
diameters = np.linspace(6.0, 8.85, 500)
M, D = np.meshgrid(masses, diameters)
grid = np.column_stack([M.ravel(), D.ravel()])
P = predict(grid).reshape(M.shape)

fig, ax = plt.subplots(figsize=(9.2, 6.3))
ax.contourf(
    M, D, P,
    levels=[0.0, 0.25, 0.50, 0.75, 1.0],
    colors=['#f3e0cb', '#ecd5c4', '#dbe7cf', '#cde3bf'],
    alpha=0.95,
)
ax.contour(M, D, P, levels=[0.5], colors=['#5a1a8a'], linewidths=2.2)

mask = y_train == 1.0
ax.scatter(X_train[mask, 0], X_train[mask, 1], marker='o', s=48,
           facecolors='white', edgecolors='black', linewidths=1.1,
           label='सेब - प्रशिक्षण')
mask = y_train == 0.0
ax.scatter(X_train[mask, 0], X_train[mask, 1], marker='s', s=48,
           facecolors='white', edgecolors='black', linewidths=1.1,
           label='संतरा - प्रशिक्षण')
mask = y_test == 1.0
ax.scatter(X_test[mask, 0], X_test[mask, 1], marker='^', s=92,
           color='#1f77b4', label='सेब - परीक्षण')
mask = y_test == 0.0
ax.scatter(X_test[mask, 0], X_test[mask, 1], marker='x', s=110,
           color='#ff7f0e', linewidths=2.0, label='संतरा - परीक्षण')

ax.set_xlim(116, 214)
ax.set_ylim(6.0, 8.85)
ax.set_xlabel('द्रव्यमान (g)')
ax.set_ylabel('व्यास (cm)')
ax.set_title('प्रशिक्षण के बाद तंत्रिका नेटवर्क का आउटपुट', fontweight='bold')
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=True)
fig.tight_layout()
plt.show()


## 5. NeuralMath

इस लेख से संबंधित सभी संगणकीय सामग्री — कोड और एक चलने योग्य Google Colab नोटबुक सहित — https://neuralmath.org पर उपलब्ध है, जहाँ पाठक उदाहरणों को देख, पुनरुत्पादित या अनुकूलित कर सकते हैं।
